# Single-photon BB84 Trial

This notebook is now a thin tutorial runner. The protocol code lives in `examples/bb84/`:

- `configs.py` holds device and post-processing settings.
- `helpers.py` holds small BB84 and timing helpers.
- `agents.py` holds the physical Alice and Bob agents.
- `trial.py` builds the two-node network and runs the trial.

The notebook stays focused on choosing parameters, running one trial, and reading the result.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "simyuj").exists() and (candidate / "examples").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not find the simyuj repository root")

PROTOCOLS_DIR = REPO_ROOT / "tutorials" / "protocols"
PROTOCOLS_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.bb84 import run_bb84_trial, summarize_trial, write_trial_report

print("Repository root:", REPO_ROOT)
print("Protocol output directory:", PROTOCOLS_DIR)


## Choose A Trial

These are realistic enough for the full pipeline while still being manageable in a notebook. The log file records the event-level simulation trace. The report file stores the final summary dictionary.


In [ ]:
trial_params = {
    "distance_km": 30.0,
    "master_seed": 2029,
    "num_slots": 30_000,
}

log_path = (
    PROTOCOLS_DIR
    / f"bb84_trial_{trial_params['distance_km']:g}km_seed{trial_params['master_seed']}.jsonl"
)
report_path = PROTOCOLS_DIR / "bb84_final_report.json"

print("Trial parameters:")
print(json.dumps(trial_params, indent=2))
print("Log path:", log_path)
print("Report path:", report_path)


## Advanced Parameter Overrides

The simple trial parameters are shortcuts. For full control, pass override dictionaries whose keys match the dataclasses in `examples/bb84/configs.py`. The default run below does not use these overrides, but this is the pattern to change timing jitter, fiber loss, dark counts, detector dead time, and other device settings.


In [ ]:
advanced_override_example = {
    "source_overrides": {
        "emission_probability": 0.55,
        "timing_jitter_stddev_s": 30e-12,
    },
    "quantum_channel_overrides": {
        "attenuation_db_per_km": 0.22,
        "fixed_insertion_loss_db": 3.0,
        "depolarizing_probability": 0.01,
    },
    "detector_overrides": {
        "efficiency": 0.75,
        "dark_count_rate_hz": 500.0,
        "dead_time_s": 80e-9,
        "jitter_stddev_s": 80e-12,
    },
    "classical_channel_overrides": {
        "fiber_speed_m_per_s": 2.0e8,
        "loss_probability": 0.0,
    },
}

print(json.dumps(advanced_override_example, indent=2))


## Run The Trial

This call creates Alice, Bob, the source, fiber channel, detector array, public classical channels, and all post-processing events.


In [ ]:
trial = run_bb84_trial(
    **trial_params,
    log_file=str(log_path),
)

write_trial_report(trial, report_path)

print(json.dumps(trial, indent=2))
print("Saved report to:", report_path)
print("Saved event log to:", log_path)


## Read The Result

The most important checks are: QBER accepted, Cascade completed, reconciled bits match, verification accepted, privacy amplification completed, and final keys match.


In [ ]:
print(summarize_trial(trial))


## What To Change Next

Good knobs to try first:

- `distance_km`: longer fiber means more loss.
- `num_slots`: more slots give more sifted bits.
- `master_seed`: changes the random run while keeping it reproducible.

For command-line runs, use `examples/bb84/demo.py`.
